# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

# Reading from bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

# Transformations

- Trim text columns
- Remove hyphen from '-' from customer id column so it is consistent and matches with other columns
- Normalize country names for Germany and United States
- Rename column headers

### Trimming text columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Clean customer ids

In [0]:
df = df.withColumn(
    "CID",
    F.regexp_replace(F.col("CID"), "-", "")
)

### Normalizing country column

In [0]:
df = df.withColumn(
    "CNTRY",
    F.when(F.col("CNTRY").isin("US", "USA", "United States"), "United States")
     .when(F.col("CNTRY").isin("DE", "Germany"), "Germany")
     .otherwise(F.col("CNTRY"))
)

### Renaming column headers

In [0]:
RENAME_MAPPING = {
    "CID": "customer_id",
    "CNTRY": "country"
}

for old_name, new_name in RENAME_MAPPING.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity check dataframe

In [0]:
df.limit(10).display()

# Writing to silver table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customer_location")

### Sanity check silver table

In [0]:
%sql

SELECT * FROM workspace.silver.erp_customer_location